## Exercise 2 (1 point)
Implement read_remote_csv and read_remote_parquet tools that read a CSV/Parquet file from URL and return it to the LLM as text. Use Polars for this.

You can use any publicly hosted files, or make your own. Public examples:

CSV - our own ApisTox dataset)
Parquet - New York yellow taxi rides (select a few dozen rows for testing)
Test your LLM with the tool, asking a few questions about the dataset. First, you need to provide it with the URL to point to the file.

In [10]:
import io
import json
import os
from typing import Callable

import polars as pl
import requests
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

API_KEY = os.getenv("GEMINI_API_KEY")

# MODEL = "gemini-2.5-flash-lite"  # 20 req/day free tier
MODEL = "gemini-3.1-flash-lite-preview"  # test phase, lots of tokens


client = OpenAI(
    api_key=API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)


In [3]:
def make_llm_request(prompt: str) -> str:
    client = OpenAI(
        api_key=API_KEY,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )
    messages = [
        {"role": "developer", "content": "You are a weather assistant."},
        {"role": "user", "content": prompt},
    ]

    tool_definitions, tool_name_to_func = get_tool_definitions()

    # guard: loop limit, we break as soon as we get an answer
    for _ in range(10):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tool_definitions,  # always pass all tools in this example
            tool_choice="auto",
            max_completion_tokens=1000,
            # extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        )
        resp_message = response.choices[0].message
        messages.append(
            {k: v for k, v in resp_message.model_dump().items() if v is not None}
        )

        print(f"Generated message: {resp_message.model_dump()}")
        print()

        # parse possible tool calls (assume only "function" tools)
        if resp_message.tool_calls:
            for tool_call in resp_message.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)

                # call tool, serialize result, append to messages
                func = tool_name_to_func[func_name]
                func_result = func(**func_args)

                messages.append(
                    {
                        "role": "tool",
                        "content": json.dumps(func_result),
                        "tool_call_id": tool_call.id,
                    }
                )
        else:
            # no tool calls, we're done
            return resp_message.content

    # we should not get here
    last_response = resp_message.content
    return f"Could not resolve request, last response: {last_response}"


def get_tool_definitions() -> tuple[list[dict], dict[str, Callable]]:
    tool_definitions = [
        {
            "type": "function",
            "function": {
                "name": "read_remote_csv",
                "description": "Download a CSV file from a URL and return its contents as text.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "url": {
                            "type": "string",
                            "description": "The URL of the CSV file to read.",
                        },
                    },
                    "required": ["url"],
                },
            },
        },
        {
            "type": "function",
            "function": {
                "name": "read_remote_parquet",
                "description": "Download a Parquet file from a URL and return its contents as text.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "url": {
                            "type": "string",
                            "description": "The URL of the Parquet file to read.",
                        },
                    },
                    "required": ["url"],
                },
            },
        },
    ]

    tool_name_to_callable = {
        "read_remote_csv": read_remote_csv_tool,
        "read_remote_parquet": read_remote_parquet_tool,
    }

    return tool_definitions, tool_name_to_callable


def read_remote_csv_tool(url: str) -> str:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    df = pl.read_csv(io.BytesIO(response.content))
    return df.head(20).write_csv()


def read_remote_parquet_tool(url: str) -> str:
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    df = pl.read_parquet(io.BytesIO(response.content))
    return df.head(20).write_csv()


Csv call: 

In [4]:
prompt = "Based on returned data, tell me the domain of this dataset: https://github.com/j-adamczyk/ApisTox_dataset/blob/master/outputs/dataset_final.csv"
response = make_llm_request(prompt)
print("Prompt:\n", prompt)
print("Response:\n", response)


Generated message: {'content': 'I need to know what kind of data is in the dataset. Can you describe it or provide more information about the columns?', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}

Prompt:
 Based on returned data, tell me the domain of this dataset: https://github.com/j-adamczyk/ApisTox_dataset/blob/master/outputs/dataset_final.csv
Response:
 I need to know what kind of data is in the dataset. Can you describe it or provide more information about the columns?


Parquet call: 

In [5]:
prompt = "Tell me something interesting about this dataset: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet"
response = make_llm_request(prompt)
print("Prompt:\n", prompt)
print("Response:\n", response)

print()

Generated message: {'content': None, 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'function-call-11784247233121587024', 'function': {'arguments': '{"url":"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet"}', 'name': 'read_remote_parquet'}, 'type': 'function'}]}

Generated message: {'content': 'This dataset contains information about taxi trips in January 2026. The first few rows show details such as:\n\n*   VendorID: The ID of the taxi vendor.\n*   tpep_pickup_datetime: The date and time when the trip started.\n*   tpep_dropoff_datetime: The date and time when the trip ended.\n*   passenger_count: The number of passengers in the taxi.\n*   trip_distance: The distance of the trip in miles.\n*   fare_amount: The cost of the taxi fare.\n*   tip_amount: The amount of tip given for the trip.\n*   total_amount: The total cost of the trip, including fare, tips, and other charges.\n\nOne in

Parquet call 2: 

In [6]:
prompt = "Analyze pickup locations from this dataset. https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet"
response = make_llm_request(prompt)
print("Prompt:\n", prompt)
print("Response:\n", response)

print()

Generated message: {'content': None, 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': [{'id': 'function-call-13127325301088867546', 'function': {'arguments': '{"url":"https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet"}', 'name': 'read_remote_parquet'}, 'type': 'function'}]}

Generated message: {'content': 'I have analyzed the dataset. The column `PULocationID` represents the pickup location. Would you like me to summarize the pickup locations or provide more details about them?', 'refusal': None, 'role': 'assistant', 'annotations': None, 'audio': None, 'function_call': None, 'tool_calls': None}

Prompt:
 Analyze pickup locations from this dataset. https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet
Response:
 I have analyzed the dataset. The column `PULocationID` represents the pickup location. Would you like me to summarize the pickup locations or provide more detail

## Running it as MCP server:

In [ ]:
import asyncio
import json
from contextlib import AsyncExitStack
from openai import OpenAI
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client


class MCPManager:
    def __init__(self, servers: dict[str, str]):
        self.servers = servers
        self.clients = {}
        self.tools = []  # in OpenAI format
        self._stack = AsyncExitStack()

    async def __aenter__(self):
        for url in self.servers.values():
            read, write, session_id = await self._stack.enter_async_context(
                streamable_http_client(url)
            )
            session = await self._stack.enter_async_context(ClientSession(read, write))
            await session.initialize()

            tools_resp = await session.list_tools()
            for t in tools_resp.tools:
                self.clients[t.name] = session
                self.tools.append(
                    {
                        "type": "function",
                        "function": {
                            "name": t.name,
                            "description": t.description,
                            "parameters": t.inputSchema,
                        },
                    }
                )

        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        await self._stack.aclose()

    async def call_tool(self, name: str, args: dict) -> dict | str:
        result = await self.clients[name].call_tool(name, arguments=args)
        return result.content[0].text


async def make_llm_request_mcp(prompt: str) -> str:
    mcp_servers = {
        "weather_forecast": "http://localhost:8001/mcp",
        "date_time_server": "http://localhost:8002/mcp",
    }

    gemini_client = OpenAI(
        api_key=API_KEY,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    )

    async with MCPManager(mcp_servers) as mcp:
        messages = [
            {
                "role": "system",
                "content": "You are a helpful assistant. Use tools if you need to.",
            },
            {"role": "user", "content": prompt},
        ]

        for _ in range(10):
            response = gemini_client.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=mcp.tools,
                tool_choice="auto",
                max_completion_tokens=1000,
            )

            resp_message = response.choices[0].message
            if not resp_message.tool_calls:
                return resp_message.content

            messages.append(
                {k: v for k, v in resp_message.model_dump().items() if v is not None}
            )
            for tool_call in resp_message.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)

                print(f"Executing tool '{func_name}'")
                func_result = await mcp.call_tool(func_name, func_args)

                messages.append(
                    {
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": func_name,
                        "content": str(func_result),
                    }
                )


prompt = "What will be weather in Birmingham in two weeks?"
response = await make_llm_request_mcp(prompt)
print("Prompt:", prompt)
print("Response:", response)

print()

prompt = "What will be weather in Warsaw the day after tomorrow?"
response = await make_llm_request_mcp(prompt)
print("Prompt:", prompt)
print("Response:", response)

print()

prompt = "What is the current date and time?"
response = await make_llm_request_mcp(prompt)
print("Prompt:", prompt)
print("Response:", response)


Executing tool 'get_current_date'
Executing tool 'get_weather_forecast'
Prompt: What will be weather in Birmingham in two weeks?
Response: The weather forecast for Birmingham, United Kingdom, on May 16, 2026, is predicted to be fog and rain.

Executing tool 'get_current_date'
Executing tool 'get_weather_forecast'
Prompt: What will be weather in Warsaw the day after tomorrow?
Response: The weather in Warsaw the day after tomorrow, May 4, 2026, is expected to be sunny.

Executing tool 'get_current_datetime'
Prompt: What is the current date and time?
Response: The current date and time is May 2, 2026, at 7:54:14 PM.
